# 03 — Dictionary Baseline

A transparent, non-neural lower bound. For each Arabic subword we look up the English
subword most often seen at the same relative position in the training data, then stitch
the guesses together. No learning, no context — it only shows how far a trivial lexical
method gets, so we can measure what the Tiny Transformer adds.

Everything (the algorithm, detokenization, BLEU/chrF++) is implemented in this notebook.

## Setup

In [1]:
import os
from collections import Counter, defaultdict

import pandas as pd
import sacrebleu
import sentencepiece as spm

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")   # run from the repo root so Data/ paths resolve

TOK = "Data/tokenized"
MAX_LEN = 80               # same length filter used for training/evaluation
sp_ar = spm.SentencePieceProcessor(model_file="Data/vocab/sp_ar.model")
sp_en = spm.SentencePieceProcessor(model_file="Data/vocab/sp_en.model")

## Load the tokenized data
The `.bpe` files are whitespace-separated SentencePiece subwords, aligned line-by-line.

In [2]:
def read_lines(path):
    return open(path, encoding="utf-8").read().splitlines()

train_ar = read_lines(f"{TOK}/train.ar.bpe")
train_en = read_lines(f"{TOK}/train.en.bpe")
test_ar = read_lines(f"{TOK}/test.ar.bpe")
test_en = read_lines(f"{TOK}/test.en.bpe")
print(f"train pairs: {len(train_ar):,} | test pairs (raw): {len(test_ar):,}")

train pairs: 50,000 | test pairs (raw): 8,567


Keep only test pairs within the length limit (same filter as the Transformer).

In [3]:
kept = [(a, e) for a, e in zip(test_ar, test_en)
        if len(a.split()) <= MAX_LEN and len(e.split()) <= MAX_LEN]
test_ar_f = [a for a, e in kept]
test_en_f = [e for a, e in kept]
print(f"test pairs kept after max_len={MAX_LEN}: {len(kept):,}")

test pairs kept after max_len=80: 8,491


## The baseline algorithm
`build_position_dictionary` maps each Arabic subword to the most frequent English subword
at the same *relative* position. `translate_baseline` applies that map token-by-token.

In [4]:
def build_position_dictionary(src_lines, tgt_lines):
    counts = defaultdict(Counter)
    for src, tgt in zip(src_lines, tgt_lines):
        s, t = src.split(), tgt.split()
        if not s or not t:
            continue
        for i, piece in enumerate(s):
            j = min(round(i * (len(t) - 1) / max(1, len(s) - 1)), len(t) - 1)
            counts[piece][t[j]] += 1
    return {piece: c.most_common(1)[0][0] for piece, c in counts.items()}

def translate_baseline(line, dictionary):
    return " ".join(dictionary.get(piece, "<unk>") for piece in line.split())

In [5]:
dictionary = build_position_dictionary(train_ar, train_en)
print(f"dictionary entries: {len(dictionary):,}")
list(dictionary.items())[:8]

dictionary entries: 8,173


[('▁اذاً', '▁so'),
 ('▁نحن', '▁we'),
 ('▁ن', '▁we'),
 ('بي', ','),
 ('عها', '▁to'),
 ('،', ','),
 ('▁ثم', '▁and'),
 ('▁شئ', '▁something')]

## Detokenization and scoring
BLEU and chrF++ are computed on **detokenized** English (SentencePiece `▁` markers removed),
so we compare natural text. chrF++ is chrF with `word_order=2`.

In [6]:
def detok(lines, sp):
    return [sp.decode(line.split()) for line in lines]

def score(hyp, ref):
    bleu = sacrebleu.corpus_bleu(hyp, [ref])
    chrfpp = sacrebleu.corpus_chrf(hyp, [ref], word_order=2)
    return bleu.score, chrfpp.score

## Translate the test set and score

In [7]:
baseline_bpe = [translate_baseline(line, dictionary) for line in test_ar_f]
baseline = detok(baseline_bpe, sp_en)
reference = detok(test_en_f, sp_en)

bleu_full, chrf_full = score(baseline, reference)
bleu_1k, chrf_1k = score(baseline[:1000], reference[:1000])
results = pd.DataFrame([
    {"split": "test_full", "examples": len(baseline), "bleu": round(bleu_full, 4), "chrf_pp": round(chrf_full, 4)},
    {"split": "test_1000", "examples": 1000, "bleu": round(bleu_1k, 4), "chrf_pp": round(chrf_1k, 4)},
])
results

,split,examples,bleu,chrf_pp
0,test_full,8491,4.7777,25.6503
1,test_1000,1000,4.7137,25.5561


In [8]:
os.makedirs("outputs/tables", exist_ok=True)
pd.DataFrame([{"model": "dictionary_position_baseline", "decoding": "position_dictionary",
               "examples": len(baseline), "bleu": round(bleu_full, 4),
               "chrf_pp": round(chrf_full, 4)}]).to_csv("outputs/tables/baseline_results.csv", index=False)
print("saved outputs/tables/baseline_results.csv")

saved outputs/tables/baseline_results.csv


## Sample translations
The output is word-salad — correct vocabulary, no grammar — exactly what a floor should look like.

In [9]:
samples = pd.DataFrame({
    "source_ar": detok(test_ar_f[:6], sp_ar),
    "reference_en": reference[:6],
    "baseline_output": baseline[:6],
})
os.makedirs("outputs/examples", exist_ok=True)
samples.to_csv("outputs/examples/baseline_samples.csv", index=False, encoding="utf-8")
samples

,source_ar,reference_en,baseline_output
0,قبل عدة سنوات، هنا في تيد، قدّم بيتر سكيلمان م...,"several years ago here at ted, peter skillman ...","before many years, here in ted,,, peter, form,..."
1,والفكرة غاية في البساطة. فريق مكوّن من اربعة ي...,and the idea's pretty simple: teams of four ha...,"and the very in simplicity. a, a of four we to..."
2,يجب ان تكون المارش مالو علي القمة.,the marshmallow has to be on top.,"we to be the the, money, on the."
3,ورغماً عن انها تبدو بسيطة للغاية، الا انها صعب...,"and, though it seems really simple, it's actua...","and the, about it look simple., the it difficu..."
4,لذا فقد فكرت بان هذه فكرة مثيرة، وقمت بتضمينها...,"and so, i thought this was an interesting idea...","so we i that this idea interesting, and to,. i..."
5,وقد كان نجاحاً باهراً.,and it was a huge success.,"and was success, the,."
